In [ ]:
from pathlib import Path

import duckdb
import polars as pl

# Lokasi folder utama proyek yang sebenarnya
PROJECT_ROOT = Path(
    r"F:\Dokumen\SEMESTER 7\Analisis Big Data\Penugasan\tugas2\tugas1-abd-2026"
)

# Folder berisi 7 file Parquet
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "data"
DATA_GLOB = str(DATA_PATH / "train-*.parquet")

# Lokasi keluaran hasil cleaning
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dataset_bersih.parquet"
)

# Periksa keberadaan folder dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Folder dataset tidak ditemukan: {DATA_PATH}"
    )

# Periksa jumlah file Parquet
files = sorted(DATA_PATH.glob("train-*.parquet"))

if not files:
    raise FileNotFoundError(
        f"Tidak ditemukan file Parquet di: {DATA_PATH}"
    )

print("Lokasi proyek :", PROJECT_ROOT)
print("Lokasi data   :", DATA_PATH)
print("Jumlah file   :", len(files))

for file in files:
    print(file.name)

# Membaca dataset secara lazy
df = pl.scan_parquet(DATA_GLOB)

jumlah_awal = df.select(pl.len()).collect().item()

print(f"\nJumlah baris awal: {jumlah_awal:,}")
print("Nama kolom:", df.collect_schema().names())

df.head(5).collect()

FileNotFoundError: Folder dataset tidak ditemukan: F:\Dokumen\SEMESTER 7\Analisis Big Data\Penugasan\tugas1-abd-2026\data\raw\data

In [ ]:
connection = duckdb.connect()

profil = connection.execute(
    f"""
    SUMMARIZE
    SELECT *
    FROM read_parquet('{DATA_GLOB}')
    """
).pl()

profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""text""","""VARCHAR""","""""","""􀂉Membentuk tenaga praktisi yan…",19907881,null,null,null,null,null,19483721,0.00


In [ ]:
jumlah_baris = jumlah_awal

# Jumlah nilai null
jumlah_null = (
    df.select(
        pl.col("text").is_null().sum().alias("jumlah_null")
    )
    .collect()
    .item()
)

# Jumlah teks kosong atau hanya whitespace, tidak termasuk null
jumlah_kosong = (
    df.filter(
        pl.col("text").is_not_null()
        & (pl.col("text").str.strip_chars() == "")
    )
    .select(pl.len())
    .collect()
    .item()
)

missing_summary = pl.DataFrame(
    {
        "jenis": ["Null", "Teks kosong/whitespace"],
        "jumlah": [jumlah_null, jumlah_kosong],
    }
).with_columns(
    (pl.col("jumlah") / jumlah_baris * 100)
    .round(4)
    .alias("persentase")
)

missing_summary

jenis,jumlah,persentase
str,i64,f64
"""Null""",0,0.0
"""Teks kosong/whitespace""",1000001,5.1325


In [ ]:
df_clean = df.filter(
    pl.col("text").is_not_null()
    & (pl.col("text").str.strip_chars() != "")
)

jumlah_setelah_missing = (
    df_clean.select(pl.len()).collect().item()
)

print(f"Baris sebelum penanganan : {jumlah_awal:,}")
print(f"Baris setelah penanganan : {jumlah_setelah_missing:,}")
print(f"Baris yang dihapus      : {jumlah_awal - jumlah_setelah_missing:,}")

Baris sebelum penanganan : 19,483,721
Baris setelah penanganan : 18,483,720
Baris yang dihapus      : 1,000,001


In [ ]:
# Hitung panjang karakter pada setiap teks yang tidak kosong
df_panjang = df_clean.select(
    pl.col("text").str.len_chars().alias("panjang_teks")
)

stat_panjang = df_panjang.select(
    pl.col("panjang_teks").quantile(0.25).alias("Q1"),
    pl.col("panjang_teks").quantile(0.75).alias("Q3"),
    pl.col("panjang_teks").median().alias("median"),
    pl.col("panjang_teks").mean().alias("rata_rata"),
    pl.col("panjang_teks").max().alias("maksimum"),
).collect()

q1 = stat_panjang["Q1"][0]
q3 = stat_panjang["Q3"][0]

iqr = q3 - q1
batas_bawah = max(0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

print(stat_panjang)
print(f"Q1          : {q1}")
print(f"Q3          : {q3}")
print(f"IQR         : {iqr}")
print(f"Batas bawah : {batas_bawah}")
print(f"Batas atas  : {batas_atas}")

shape: (1, 5)
┌──────┬───────┬────────┬────────────┬──────────┐
│ Q1   ┆ Q3    ┆ median ┆ rata_rata  ┆ maksimum │
│ ---  ┆ ---   ┆ ---    ┆ ---        ┆ ---      │
│ f64  ┆ f64   ┆ f64    ┆ f64        ┆ u32      │
╞══════╪═══════╪════════╪════════════╪══════════╡
│ 46.0 ┆ 224.0 ┆ 96.0   ┆ 170.083601 ┆ 180383   │
└──────┴───────┴────────┴────────────┴──────────┘
Q1          : 46.0
Q3          : 224.0
IQR         : 178.0
Batas bawah : 0
Batas atas  : 491.0


In [ ]:
jumlah_outlier = (
    df_panjang.filter(
        (pl.col("panjang_teks") < batas_bawah)
        | (pl.col("panjang_teks") > batas_atas)
    )
    .select(pl.len())
    .collect()
    .item()
)

print(f"Jumlah teks yang terdeteksi sebagai outlier: {jumlah_outlier:,}")

Jumlah teks yang terdeteksi sebagai outlier: 800,291


In [ ]:
# Hitung jumlah nilai teks unik sebelum deduplikasi
jumlah_teks_unik = (
    df_clean.select(pl.col("text").n_unique())
    .collect()
    .item()
)

jumlah_duplikat = jumlah_setelah_missing - jumlah_teks_unik

print(f"Jumlah baris setelah penanganan missing : {jumlah_setelah_missing:,}")
print(f"Jumlah teks unik                       : {jumlah_teks_unik:,}")
print(f"Jumlah duplikat                        : {jumlah_duplikat:,}")

Jumlah baris setelah penanganan missing : 18,483,720
Jumlah teks unik                       : 18,483,720
Jumlah duplikat                        : 0


In [ ]:
df_dedup = df_clean.unique(
    subset=["text"],
    keep="first",
    maintain_order=False,
)

print("Transformasi deduplikasi telah disiapkan.")

Transformasi deduplikasi telah disiapkan.


In [ ]:
# Validasi pada hasil setelah deduplikasi
validasi = df_dedup.select(
    pl.len().alias("total_baris"),
    pl.col("text").is_null().sum().alias("jumlah_null"),
    (
        pl.col("text").is_not_null()
        & (pl.col("text").str.strip_chars() == "")
    ).sum().alias("jumlah_teks_kosong"),
    pl.col("text").n_unique().alias("jumlah_teks_unik"),
).collect()

total_baris_bersih = validasi["total_baris"][0]
jumlah_null_akhir = validasi["jumlah_null"][0]
jumlah_kosong_akhir = validasi["jumlah_teks_kosong"][0]
jumlah_unik_akhir = validasi["jumlah_teks_unik"][0]

jumlah_duplikat_akhir = total_baris_bersih - jumlah_unik_akhir

validasi

total_baris,jumlah_null,jumlah_teks_kosong,jumlah_teks_unik
u32,u32,u32,u32
18483720,0,0,18483720


In [ ]:
assert jumlah_null_akhir == 0, "Masih ditemukan nilai null."
assert jumlah_kosong_akhir == 0, "Masih ditemukan teks kosong."
assert jumlah_duplikat_akhir == 0, "Masih ditemukan teks duplikat."

print("Semua aturan validasi berhasil dipenuhi.")

Semua aturan validasi berhasil dipenuhi.


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Simpan hasil cleaning ke disk.
# Proses ini akan menjalankan transformasi LazyFrame.
df_dedup.sink_parquet(OUTPUT_PATH)

print(f"Dataset bersih tersimpan di: {OUTPUT_PATH}")

Dataset bersih tersimpan di: F:\Dokumen\SEMESTER 7\Analisis Big Data\Penugasan\tugas1-abd-2026\data\processed\dataset_bersih.parquet


In [ ]:
df_hasil = pl.scan_parquet(str(OUTPUT_PATH))

validasi_akhir = df_hasil.select(
    pl.len().alias("total_baris"),
    pl.col("text").is_null().sum().alias("jumlah_null"),
    (
        pl.col("text").is_not_null()
        & (pl.col("text").str.strip_chars() == "")
    ).sum().alias("jumlah_teks_kosong"),
    pl.col("text").n_unique().alias("jumlah_teks_unik"),
).collect()

validasi_akhir

total_baris,jumlah_null,jumlah_teks_kosong,jumlah_teks_unik
u32,u32,u32,u32
18483720,0,0,18483720


In [ ]:
jumlah_akhir = validasi_akhir["total_baris"][0]

print(f"Baris sebelum cleaning : {jumlah_awal:,}")
print(f"Baris sesudah cleaning : {jumlah_akhir:,}")
print(f"Total baris berkurang  : {jumlah_awal - jumlah_akhir:,}")
print(f"File keluaran          : {OUTPUT_PATH}")

Baris sebelum cleaning : 19,483,721
Baris sesudah cleaning : 18,483,720
Total baris berkurang  : 1,000,001
File keluaran          : F:\Dokumen\SEMESTER 7\Analisis Big Data\Penugasan\tugas1-abd-2026\data\processed\dataset_bersih.parquet
